# Zero-replacement test

> "Please report the fold-change selection and SHAP order under an alternative zero replacement, or state in the abstract and at lines 437-444 that the selected taxa reflect detection versus non-detection under the chosen pseudocount."

The main pipeline replaces zero relative abundances with a fixed pseudocount of 1 × 10⁻⁶ before the CLR transformation (`1. preprocessing.ipynb`, Section 5).
The reviewer reported that replacing zeros instead with **half of the smallest non-zero relative abundance within each sample** changes the seed-42 fold-change ranking
(Brochothrix 18th → 1st, Acinetobacter 37th → 6th, Pseudomonas 53rd → 12th, Iodobacter 1st → 2nd).

This notebook repeats the analysis under both zero replacements with identical code, so that any difference comes only from the zero replacement.
Feature selection and model performance are compared for all 4 conditions × 5 models; holdout validation and SHAP are compared for the final model (Condition 1 × Random Forest).
The main pipeline is not changed: all reported results keep the 1 × 10⁻⁶ pseudocount.

| Section | Content |
|---|---|
| 1–2 | Functions, data load |
| 3 | Description of the data: abundance scale, smallest non-zero values, non-detection rates |
| 4 | CLR under both zero replacements; check that the current scheme reproduces `genus_preprocessed.pkl` |
| 5 | Fold-change ranking (seed 42 Top 20, Top 10 for all seeds) and non-detection in the selected genera |
| 6 | Feature selection for all 4 conditions (fold-change Top 10 and VIP ≥ 1.0) |
| 7 | Final model (Condition 1 × Random Forest): CV test and holdout performance |
| 8 | All 4 conditions × 5 models: CV test performance, classification and regression |
| 9 | SHAP rankings of the final model: classification and regression, test and validation sets |
| 10 | Save |

Labels are defined by TPC thresholds only and are therefore the same under both zero replacements (`labeled_splits.pkl` and `kmeans_labeled_splits.pkl` are reused).
Results for the current scheme are read from the outputs of notebooks 4 and 5; only the alternative scheme is trained here (plus the final model under both schemes, for holdout and SHAP).
Differences in performance are tested with a paired t-test (pairs = seeds, n = 10).

**Run after notebooks 1–6** (this notebook reads their saved outputs).
Outputs: `output/pkl/zero_replacement_results.pkl`, `output/pkl/zero_replacement_grid.pkl` and `output/zero_replacement_test.xlsx`

In [1]:
import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ── TensorFlow 로그 억제 (5. regression.ipynb 와 동일) ─────────────
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
_fd, _saved = os.open(os.devnull, os.O_WRONLY), os.dup(2)
os.dup2(_fd, 2)
try:
    import tensorflow as tf
finally:
    os.dup2(_saved, 2); os.close(_saved); os.close(_fd)
tf.get_logger().setLevel('ERROR')
try:
    import absl.logging
    absl.logging.set_verbosity(absl.logging.ERROR)
except ImportError:
    pass

import shap
import sklearn.base as _base
from scipy import stats
from scipy.stats import f_oneway
from statsmodels.stats.multitest import multipletests
from sklearn.cross_decomposition import PLSRegression
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, r2_score, mean_absolute_error
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMRegressor

# Repository-relative paths. Run this notebook from the repository root.
OUTPUT_DIR = "output"
PKL_DIR    = f"{OUTPUT_DIR}/pkl"

SEEDS       = [38, 39, 40, 41, 42, 43, 44, 45, 46, 47]
LABEL_SEED  = 42
LABEL_ORDER = [0, 2, 1]   # Early → Intermediate → Late
PSEUDOCOUNT = 1e-6        # 1. preprocessing.ipynb 의 주 파이프라인 값

SCHEMES = {
    'current':     'Pseudocount 1e-6 (main pipeline)',
    'alternative': 'Half of the smallest non-zero value in each sample',
}

# 리뷰어가 보고한 seed-42 fold-change 순위 (현행 → 대안)
REVIEWER_RANKS = {'Brochothrix': (18, 1), 'Acinetobacter': (37, 6),
                  'Pseudomonas': (53, 12), 'Iodobacter': (1, 2)}
KEY_TAXA = ['Iodobacter', 'Vagococcus', 'Streptococcus']   # 본문에서 핵심 taxa 로 언급한 genus

# Random Forest — 4. classification.ipynb / 5. regression.ipynb 와 같은 설정 (Section 6 에서 재현 확인)
RF_GRID = {'n_estimators': [100, 200], 'max_depth': [3, 5, 7],
           'min_samples_split': [2, 5]}

CONDS = ['cond1', 'cond2', 'cond3', 'cond4']
COND_LABELS = {
    'cond1': 'Optuna × FC Top10',
    'cond2': 'Optuna × VIP≥1.0',
    'cond3': 'K-means × FC Top10',
    'cond4': 'K-means × VIP≥1.0',
}
COND_SCREEN_KEY = {'cond1': 'cond1_optuna_fc10', 'cond2': 'cond2_optuna_vip',
                   'cond3': 'cond3_kmeans_fc10', 'cond4': 'cond4_kmeans_vip'}
COND_LABELING   = {'cond1': 'optuna', 'cond2': 'optuna',
                   'cond3': 'kmeans', 'cond4': 'kmeans'}
FINAL_COND, FINAL_MODEL = 'cond1', 'RandomForest'

## 1. Functions

- `clr_with_iqr_capping`: zero replacement → CLR → per-day IQR capping, as in `1. preprocessing.ipynb` Sections 5–6. Only the zero replacement can be changed.
- `anova_fdr`, `vip_select`, `fold_change_top10`: copied without modification from `3. screening.ipynb`.
- `clf_configs`, `LSTMRegressor`, `reg_configs`: copied without modification from `4. classification.ipynb` and `5. regression.ipynb` (`model_configs`).

In [2]:
def extract_day(desc):
    m = re.match(r'A(\d+)-\d+', str(desc))
    return int(m.group(1)) if m else np.nan


def replace_zeros(X, scheme):
    """X: 시료 × genus 상대풍부도(%) 배열"""
    X = X.astype(np.float64).copy()
    if scheme == 'current':
        X[X == 0] = PSEUDOCOUNT
    elif scheme == 'alternative':
        for i in range(X.shape[0]):
            row = X[i]
            row[row == 0] = 0.5 * row[row > 0].min()
    else:
        raise ValueError(scheme)
    return X


def clr_with_iqr_capping(raw, feats, scheme):
    """0 대체 → CLR → 저장일별 IQR capping (1. preprocessing.ipynb Section 5–6 과 동일)"""
    X     = replace_zeros(raw[feats].values, scheme)
    log_X = np.log(X)
    clr_X = log_X - log_X.mean(axis=1, keepdims=True)

    df = raw[['Description']].copy()
    df[feats] = clr_X
    df['Day'] = df['Description'].apply(extract_day)
    for feat in feats:
        for day, group in df.groupby('Day'):
            vals = group[feat].dropna()
            if len(vals) < 4:
                continue
            q1, q3 = np.percentile(vals, [25, 75])
            iqr = q3 - q1
            lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            idxs = group.index
            df.loc[idxs, feat] = df.loc[idxs, feat].clip(lower=lower, upper=upper)
    return df.drop(columns=['Day'])


def build_splits(clr_df, src_splits):
    """주 파이프라인과 같은 분할·라벨·TPC 에 CLR 값만 바꿔 끼운다."""
    clr_idx = clr_df.set_index('Description')
    out = {}
    for seed in SEEDS:
        out[seed] = {}
        for part in ('train', 'test', 'val'):
            ref = src_splits[seed][part]
            d = clr_idx.loc[ref['Description'].values, feature_cols].reset_index()
            d['Label'] = ref['Label'].values
            d[tpc_col] = ref[tpc_col].values
            out[seed][part] = d
    return out



def select_features(df_train, cond):
    """3. screening.ipynb 와 같은 방식으로 condition 별 feature 를 선택한다."""
    if cond in ('cond1', 'cond3'):
        feats, _ = fold_change_top10(df_train, feature_cols)
        return feats
    anova_feats, _ = anova_fdr(df_train, feature_cols)
    feats, _, _ = vip_select(df_train[feature_cols], df_train['Label'].values, anova_feats)
    return feats


def anova_fdr(df_train, feature_cols, alpha=0.05):
    """Day-based one-way ANOVA + BH FDR (q < alpha). Train only."""
    df = df_train.copy()
    if 'Day' not in df.columns:
        df['Day'] = df['Description'].apply(extract_day)

    p_values, feats_valid = [], []
    for feat in feature_cols:
        groups = [g[feat].dropna().values for _, g in df.groupby('Day')]
        valid  = [g for g in groups if len(g) > 1]
        if len(valid) >= 2:
            _, p = f_oneway(*valid)
            p_values.append(p)
            feats_valid.append(feat)

    if not p_values:
        return [], pd.DataFrame(columns=['feature', 'p_value', 'q_value', 'significant'])

    p_arr = np.array(p_values)
    reject, q_arr, _, _ = multipletests(p_arr, method='fdr_bh', alpha=alpha)
    result_df = pd.DataFrame({
        'feature':     feats_valid,
        'p_value':     p_arr,
        'q_value':     q_arr,
        'significant': reject,
    }).sort_values('q_value').reset_index(drop=True)
    sig_feats = result_df.loc[result_df['significant'], 'feature'].tolist()
    return sig_feats, result_df


def vip_select(X_train, y_train, anova_features, threshold=1.0):
    """ANOVA pre-filter → PLS-DA VIP ≥ threshold. Train only."""
    X_in    = X_train[anova_features] if len(anova_features) >= 2 else X_train
    classes = sorted(np.unique(y_train))
    y_oh    = np.zeros((len(y_train), len(classes)))
    for i, c in enumerate(classes):
        y_oh[y_train == c, i] = 1.0
    n_comp = min(2, X_in.shape[1], len(classes) - 1)
    pls    = PLSRegression(n_components=n_comp, scale=False)
    pls.fit(X_in.values, y_oh)
    t, w, q_ = pls.x_scores_, pls.x_weights_, pls.y_loadings_
    ss        = np.sum(q_**2, axis=0) * np.sum(t**2, axis=0)
    vip_scores = np.sqrt(X_in.shape[1] * np.sum(w**2 * ss, axis=1) / np.sum(ss))
    vip_df = pd.DataFrame({
        'feature':   X_in.columns,
        'vip_score': vip_scores,
    }).sort_values('vip_score', ascending=False).reset_index(drop=True)
    selected = vip_df.loc[vip_df['vip_score'] >= threshold, 'feature'].tolist()
    return selected, vip_df, pls


def fold_change_top10(df_train, feature_cols, top_n=10):
    """Day별 CLR mean의 max − min (원본 방식). Train only."""
    df = df_train.copy()
    if 'Day' not in df.columns:
        df['Day'] = df['Description'].apply(extract_day)

    day_mean = df.groupby('Day')[feature_cols].mean()
    days     = sorted(day_mean.index)

    abs_change = day_mean.max() - day_mean.min()
    direction  = day_mean.loc[days[-1]] - day_mean.loc[days[0]]

    fc_df = pd.DataFrame({
        'feature':    feature_cols,
        'abs_change': abs_change.values,
        'direction':  direction.values,
    }).sort_values('abs_change', ascending=False).reset_index(drop=True)

    return fc_df['feature'].iloc[:top_n].tolist(), fc_df

In [3]:
clf_configs = {
    "RandomForest": {
        "estimator": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200],
            "max_depth":    [3, 5, 7],
            "min_samples_split": [2, 5],
        },
    },
    "XGBoost": {
        "estimator": XGBClassifier(random_state=42, eval_metric='mlogloss',
                                   verbosity=0, use_label_encoder=False),
        "params": {
            "n_estimators":  [100, 200],
            "max_depth":     [3, 5, 7],
            "learning_rate": [0.05, 0.1],
        },
    },
    "SVC": {
        "estimator": Pipeline([('sc', StandardScaler()),
                               ('svc', SVC(probability=True, class_weight='balanced'))]),
        "params": {
            "svc__C":      [0.5, 1, 2],
            "svc__gamma":  ["scale", "auto"],
            "svc__kernel": ["rbf"],
        },
    },
    "LogisticRegression": {
        "estimator": Pipeline([('sc', StandardScaler()),
                               ('lr', LogisticRegression(penalty='elasticnet', solver='saga',
                                                         max_iter=500, class_weight='balanced',
                                                         random_state=42))]),
        "params": {
            "lr__l1_ratio": [0.2, 0.5, 0.8],
            "lr__C":        [0.5, 1, 2],
        },
    },
    "KNN": {
        "estimator": KNeighborsClassifier(),
        "params": {
            "n_neighbors": [3, 5, 7],
            "weights":     ["uniform", "distance"],
        },
    },
}

In [4]:
class LSTMRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, units=32, dropout=0.2, epochs=100,
                 batch_size=16, learning_rate=0.001, random_state=42):
        self.units         = units
        self.dropout       = dropout
        self.epochs        = epochs
        self.batch_size    = batch_size
        self.learning_rate = learning_rate
        self.random_state  = random_state

    def _build(self, n_feat):
        tf.random.set_seed(self.random_state)
        m = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(1, n_feat)),
            tf.keras.layers.LSTM(self.units, return_sequences=False),
            tf.keras.layers.Dropout(self.dropout),
            tf.keras.layers.Dense(16, activation='relu'),
            tf.keras.layers.Dense(1),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(self.learning_rate), loss='mae')
        return m

    def fit(self, X, y):
        np.random.seed(self.random_state)
        tf.random.set_seed(self.random_state)
        self.n_features_ = X.shape[1]
        self.model_ = self._build(self.n_features_)
        self.model_.fit(X.reshape(-1, 1, self.n_features_), y,
                        epochs=self.epochs, batch_size=self.batch_size, verbose=0)
        return self

    def predict(self, X):
        return self.model_.predict(
            X.reshape(-1, 1, X.shape[1]), verbose=0).flatten()

reg_configs = {
    "LightGBM": {
        "estimator": LGBMRegressor(random_state=42, verbose=-1),
        "params": {
            "n_estimators":  [100, 200],
            "max_depth":     [3, 5, 7],
            "learning_rate": [0.05, 0.1],
            "num_leaves":    [20, 31],
        },
    },
    "RandomForest": {
        "estimator": RandomForestRegressor(random_state=42),
        "params": {
            "n_estimators":      [100, 200],
            "max_depth":         [3, 5, 7],
            "min_samples_split": [2, 5],
        },
    },
    "XGBoost": {
        "estimator": XGBRegressor(random_state=42, eval_metric='mae', verbosity=0),
        "params": {
            "n_estimators":  [100, 200],
            "max_depth":     [3, 5, 7],
            "learning_rate": [0.05, 0.1],
        },
    },
    "SVR": {
        "estimator": Pipeline([('sc', StandardScaler()), ('svr', SVR())]),
        "params": {
            "svr__C":      [0.1, 1, 10],
            "svr__gamma":  ["scale", "auto"],
            "svr__kernel": ["rbf", "linear"],
        },
    },
}

reg_configs['LSTM'] = {
    'estimator': LSTMRegressor(random_state=42),
    'params': {
        'units':         [32, 64],
        'dropout':       [0.1, 0.2],
        'epochs':        [100, 150],
        'learning_rate': [0.001, 0.005],
    },
}

## 2. Data load

- `genus_rare_filtered.pkl` — relative abundance (%) after imputation and rare-feature filtering, zeros kept (`1. preprocessing.ipynb`)
- `genus_preprocessed.pkl` — main-pipeline CLR data, used to check Section 4
- `labeled_splits.pkl`, `kmeans_labeled_splits.pkl` — train/test/validation partitions and labels of the main pipeline
- `classification_results.pkl`, `regression_results.pkl` — main-pipeline results for all conditions and models (current scheme)
- `screening_results.pkl`, `sensitivity_results.pkl`, `reg_sensitivity_results.pkl`, `validation_results.pkl` — main-pipeline results, used for the checks

In [5]:
rare_path = f"{PKL_DIR}/genus_rare_filtered.pkl"
assert os.path.exists(rare_path), "genus_rare_filtered.pkl 이 없습니다 — 1. preprocessing.ipynb 를 먼저 실행하세요"

with open(rare_path, 'rb') as f:
    rare = pickle.load(f)
with open(f"{PKL_DIR}/genus_preprocessed.pkl", 'rb') as f:
    gp = pickle.load(f)
with open(f"{PKL_DIR}/labeled_splits.pkl", 'rb') as f:
    lab = pickle.load(f)
with open(f"{PKL_DIR}/screening_results.pkl", 'rb') as f:
    screen = pickle.load(f)
with open(f"{PKL_DIR}/sensitivity_results.pkl", 'rb') as f:
    sens_clf = pickle.load(f)['RandomForest']
with open(f"{PKL_DIR}/reg_sensitivity_results.pkl", 'rb') as f:
    sens_reg = pickle.load(f)['RandomForest']
with open(f"{PKL_DIR}/validation_results.pkl", 'rb') as f:
    val_res = pickle.load(f)
with open(f"{PKL_DIR}/kmeans_labeled_splits.pkl", 'rb') as f:
    km = pickle.load(f)
with open(f"{PKL_DIR}/classification_results.pkl", 'rb') as f:
    stored_clf = pickle.load(f)
with open(f"{PKL_DIR}/regression_results.pkl", 'rb') as f:
    stored_reg = pickle.load(f)

raw          = rare['data'].reset_index(drop=True)
feature_cols = rare['feature_cols']
lab_splits   = lab['labeled_splits']
km_splits    = km['labeled_splits']
tpc_col      = lab['tpc_col']

assert list(feature_cols) == list(gp['feature_cols']) == list(lab['feature_cols'])
print(f"samples: {len(raw)} | genera: {len(feature_cols)} | tpc_col: {tpc_col}")

samples: 270 | genera: 110 | tpc_col: TPC (log CFU/g)


## 3. Description of the data

Values that determine how strongly the pseudocount affects the CLR values: the abundance scale, the smallest observed non-zero abundances,
how far below them the pseudocount lies, and how often each genus is not detected.

The values are computed on all 270 samples. The 11 samples imputed with day-wise means in `1. preprocessing.ipynb` (Section 2) have the 11 smallest per-sample values (0.000295–0.001077%);
the smallest non-zero relative abundance among the 259 measured samples is 0.00192%.

In [6]:
V   = raw[feature_cols].values.astype(np.float64)
nz  = V[V > 0]
row_sum        = V.sum(axis=1)
per_sample_min = np.array([r[r > 0].min() for r in V])
alt_values     = 0.5 * per_sample_min

if np.allclose(row_sum, 100):
    scale = '0–100 (%)'
elif np.allclose(row_sum, 1):
    scale = '0–1'
else:
    scale = f'other (row sums {row_sum.min():.4g}–{row_sum.max():.4g})'

df_desc = pd.DataFrame([
    ('Relative abundance scale', scale),
    ('Row sum, min', row_sum.min()),
    ('Row sum, max', row_sum.max()),
    ('Largest relative abundance (%)', V.max()),
    ('Zero entries', int((V == 0).sum())),
    ('All entries', int(V.size)),
    ('Zero fraction', (V == 0).mean()),
    ('Smallest non-zero relative abundance, all samples (%)', nz.min()),
    ('Per-sample smallest non-zero (%), min', per_sample_min.min()),
    ('Per-sample smallest non-zero (%), median', np.median(per_sample_min)),
    ('Per-sample smallest non-zero (%), max', per_sample_min.max()),
    ('Pseudocount, main pipeline', PSEUDOCOUNT),
    ('Smallest non-zero / pseudocount', nz.min() / PSEUDOCOUNT),
    ('log10(smallest non-zero / pseudocount)', np.log10(nz.min() / PSEUDOCOUNT)),
    ('Median per-sample smallest non-zero / pseudocount', np.median(per_sample_min) / PSEUDOCOUNT),
    ('log10(median per-sample smallest non-zero / pseudocount)', np.log10(np.median(per_sample_min) / PSEUDOCOUNT)),
    ('Alternative replacement value (%), min', alt_values.min()),
    ('Alternative replacement value (%), median', np.median(alt_values)),
    ('Alternative replacement value (%), max', alt_values.max()),
], columns=['item', 'value'])
print(df_desc.to_string(index=False))

# ── genus 별 비검출률 (110 genus, 전체 270 시료) ─────────────────
day   = raw['Description'].apply(extract_day).values
early = day < 9
nd_rows = []
for j, g in enumerate(feature_cols):
    x = V[:, j]
    nd_rows.append({'genus': g.replace('g__', ''),
                    'nondetection_all':     (x == 0).mean(),
                    'nondetection_day_lt9': (x[early] == 0).mean(),
                    'nondetection_day_ge9': (x[~early] == 0).mean(),
                    'mean_abundance_pct':   x.mean(),
                    'max_abundance_pct':    x.max()})
df_nondetect = (pd.DataFrame(nd_rows)
                .sort_values('nondetection_all', ascending=False).reset_index(drop=True))

print()
print("=== Non-detection rate of the 110 genera (all 270 samples) ===")
print(df_nondetect['nondetection_all'].describe().round(3).to_string())
for t in (0.1, 0.25, 0.5, 0.75, 0.9):
    print(f"  genera with non-detection > {t:.0%}: {(df_nondetect['nondetection_all'] > t).sum()}")

# ── 주 파이프라인 FC Top10 의 비검출률 (각 seed 학습셋) ─────────────
fc_nd_rows = []
for seed in SEEDS:
    tr_desc = set(lab_splits[seed]['train']['Description'])
    in_tr   = raw['Description'].isin(tr_desc).values
    for rank, g in enumerate(screen['cond1_optuna_fc10']['per_seed_features'][seed], start=1):
        x = V[in_tr, feature_cols.index(g)]
        fc_nd_rows.append({'seed': seed, 'fc_rank': rank, 'genus': g.replace('g__', ''),
                           'nondetection_train': (x == 0).mean(),
                           'mean_abundance_pct_train': x.mean()})
df_fc_nondetect = pd.DataFrame(fc_nd_rows)

print()
print("=== Main-pipeline FC Top10, seed 42: non-detection in the training set ===")
print(df_fc_nondetect[df_fc_nondetect['seed'] == LABEL_SEED].round(4).to_string(index=False))
print()
print("=== Main-pipeline FC Top10, mean non-detection per seed (training set) ===")
print(df_fc_nondetect.groupby('seed')['nondetection_train'].mean().round(3).to_string())

                                                    item                      value
                                Relative abundance scale other (row sums 84.63–100)
                                            Row sum, min                    84.6302
                                            Row sum, max                      100.0
                          Largest relative abundance (%)                  83.751651
                                            Zero entries                      21145
                                             All entries                      29700
                                           Zero fraction                   0.711953
   Smallest non-zero relative abundance, all samples (%)                   0.000295
                   Per-sample smallest non-zero (%), min                   0.000295
                Per-sample smallest non-zero (%), median                   0.020883
                   Per-sample smallest non-zero (%), max                    

## 4. CLR under both zero replacements

The current scheme must reproduce the main-pipeline CLR values (`genus_preprocessed.pkl`) exactly; the notebook stops otherwise.

In [7]:
clr_data = {s: clr_with_iqr_capping(raw, feature_cols, s) for s in SCHEMES}

ref = gp['data'].set_index('Description')[feature_cols]
cur = clr_data['current'].set_index('Description').loc[ref.index, feature_cols]
max_diff_clr = float(np.abs(cur.values - ref.values).max())
print(f"current scheme vs genus_preprocessed.pkl: max |difference| = {max_diff_clr:.2e}")
assert max_diff_clr < 1e-9, "현행 방식 CLR 이 genus_preprocessed.pkl 을 재현하지 못함 — 중단"

splits = {s: build_splits(clr_data[s], lab_splits) for s in SCHEMES}          # Optuna 라벨
splits_by_labeling = {s: {'optuna': splits[s], 'kmeans': build_splits(clr_data[s], km_splits)}
                      for s in SCHEMES}

for s in SCHEMES:
    v = clr_data[s][feature_cols].values
    print(f"{s:12s} CLR range: {v.min():8.3f} ~ {v.max():7.3f}")

current scheme vs genus_preprocessed.pkl: max |difference| = 6.04e-14
current      CLR range:   -9.569 ~  17.257
alternative  CLR range:   -3.070 ~   9.816


## 5. Fold-change ranking

Fold change as in `3. screening.ipynb`: max − min of the day-wise mean CLR values in each seed's training set.
The current scheme must reproduce the stored Condition 1 features (`screening_results.pkl`) for all 10 seeds.

The last table reports, for each seed's Top 10, how many genera are never detected on at least one storage day of the training set
(a day-wise mean of exactly zero), and how many have at least one non-detection on the day with the lowest mean CLR value.

In [8]:
fc = {s: {seed: fold_change_top10(splits[s][seed]['train'], feature_cols) for seed in SEEDS}
      for s in SCHEMES}

for seed in SEEDS:
    assert fc['current'][seed][0] == screen['cond1_optuna_fc10']['per_seed_features'][seed], \
        f"seed {seed}: 현행 방식 FC Top10 이 screening_results.pkl 과 다름 — 중단"
print("current scheme FC Top10 = screening_results.pkl (10 seeds)")

short = lambda g: g.replace('g__', '')

# ── seed 42 전체 순위 (110 genus) ─────────────────────────────────
rk = {}
for s in SCHEMES:
    d = fc[s][LABEL_SEED][1].copy()
    d['rank'] = np.arange(1, len(d) + 1)
    rk[s] = d.set_index('feature')
df_fc_rank42 = pd.DataFrame({
    'genus':                   [short(g) for g in feature_cols],
    'abs_change_current':      [rk['current'].loc[g, 'abs_change']     for g in feature_cols],
    'rank_current':            [rk['current'].loc[g, 'rank']           for g in feature_cols],
    'abs_change_alternative':  [rk['alternative'].loc[g, 'abs_change'] for g in feature_cols],
    'rank_alternative':        [rk['alternative'].loc[g, 'rank']       for g in feature_cols],
}).sort_values('rank_current').reset_index(drop=True)

# ── seed 42 Top 20 나란히 ─────────────────────────────────────────
top20 = []
for r in range(1, 21):
    gc = rk['current'].index[rk['current']['rank'] == r][0]
    ga = rk['alternative'].index[rk['alternative']['rank'] == r][0]
    top20.append({'rank': r,
                  'current_genus': short(gc),
                  'current_abs_change': rk['current'].loc[gc, 'abs_change'],
                  'current_genus_rank_in_alternative': rk['alternative'].loc[gc, 'rank'],
                  'alternative_genus': short(ga),
                  'alternative_abs_change': rk['alternative'].loc[ga, 'abs_change'],
                  'alternative_genus_rank_in_current': rk['current'].loc[ga, 'rank']})
df_fc_top20 = pd.DataFrame(top20)

rho = stats.spearmanr(df_fc_rank42['rank_current'], df_fc_rank42['rank_alternative']).statistic
print()
print("=== Seed 42 fold-change ranking, Top 20 ===")
print(df_fc_top20.round(3).to_string(index=False))
print(f"\nSpearman rank correlation over all {len(feature_cols)} genera (seed 42): {rho:.4f}")

# ── 리뷰어가 보고한 순위 ───────────────────────────────────────────
rev_rows = []
for g, (r_cur, r_alt) in REVIEWER_RANKS.items():
    row = df_fc_rank42[df_fc_rank42['genus'] == g].iloc[0]
    rev_rows.append({'genus': g,
                     'reviewer_rank_current': r_cur, 'rank_current': int(row['rank_current']),
                     'reviewer_rank_alternative': r_alt, 'rank_alternative': int(row['rank_alternative']),
                     'reproduced': (r_cur == row['rank_current']) and (r_alt == row['rank_alternative'])})
df_fc_reviewer = pd.DataFrame(rev_rows)
print()
print("=== Ranks reported by the reviewer (seed 42) ===")
print(df_fc_reviewer.to_string(index=False))

# ── 10 seed FC Top10 ──────────────────────────────────────────────
t10_rows = []
for seed in SEEDS:
    a, b = fc['current'][seed][0], fc['alternative'][seed][0]
    t10_rows.append({'seed': seed,
                     'top10_current':     ', '.join(short(g) for g in a),
                     'top10_alternative': ', '.join(short(g) for g in b),
                     'n_shared': len(set(a) & set(b)),
                     'only_current':     ', '.join(short(g) for g in a if g not in b),
                     'only_alternative': ', '.join(short(g) for g in b if g not in a)})
df_fc_top10 = pd.DataFrame(t10_rows)
print()
print("=== FC Top10, all seeds ===")
print(df_fc_top10[['seed', 'n_shared', 'only_current', 'only_alternative']].to_string(index=False))

# ── Top10 의 비검출 진단 ──────────────────────────────────────────
diag_rows = []
raw_idx = raw.set_index('Description')
for s in SCHEMES:
    for seed in SEEDS:
        tr    = splits[s][seed]['train']
        feats = fc[s][seed][0]
        rtr   = raw_idx.loc[tr['Description'].values, feats].copy()
        rtr['Day'] = [extract_day(d) for d in tr['Description']]
        day_mean_raw = rtr.groupby('Day')[feats].mean()
        day_nd       = rtr.groupby('Day')[feats].apply(lambda x: (x == 0).mean())
        ctr = tr[feats].copy()
        ctr['Day'] = rtr['Day'].values
        min_day = ctr.groupby('Day')[feats].mean().idxmin()
        diag_rows.append({
            'scheme': s, 'seed': seed,
            'n_with_zero_day_mean':        int((day_mean_raw == 0).any(axis=0).sum()),
            'n_nondetected_on_min_clr_day': int(sum(day_nd.loc[min_day[g], g] > 0 for g in feats)),
            'mean_nondetection_train':     float((rtr[feats] == 0).mean().mean()),
        })
df_fc_diag = pd.DataFrame(diag_rows)
print()
print("=== Non-detection in the FC Top10 (training set) ===")
print(df_fc_diag.pivot(index='seed', columns='scheme',
                       values=['n_with_zero_day_mean', 'n_nondetected_on_min_clr_day',
                               'mean_nondetection_train']).round(3).to_string())

current scheme FC Top10 = screening_results.pkl (10 seeds)

=== Seed 42 fold-change ranking, Top 20 ===
 rank                  current_genus  current_abs_change  current_genus_rank_in_alternative              alternative_genus  alternative_abs_change  alternative_genus_rank_in_current
    1                     Iodobacter              15.271                                  2                    Brochothrix                   6.538                                 18
    2                 Carnobacterium              14.773                                  9                     Iodobacter                   5.306                                  1
    3                     Vagococcus              14.453                                  4              Janthinobacterium                   4.757                                 13
    4         Hafnia-Obesumbacterium              13.734                                  7                     Vagococcus                   4.678                      

## 6. Feature selection — all conditions

Feature selection of `3. screening.ipynb`, repeated on the CLR values of each zero replacement:
fold-change Top 10 (Conditions 1 and 3) and day-wise ANOVA → PLS-DA VIP ≥ 1.0 (Conditions 2 and 4).
Labels are the main-pipeline labels in both schemes.
The current scheme must reproduce `screening_results.pkl` for all 4 conditions × 10 seeds.

In [9]:
features = {s: {c: {} for c in CONDS} for s in SCHEMES}
fs_rows = []
for c in CONDS:
    for seed in SEEDS:
        for s in SCHEMES:
            tr = splits_by_labeling[s][COND_LABELING[c]][seed]['train']
            features[s][c][seed] = select_features(tr, c)
        ref = screen[COND_SCREEN_KEY[c]]['per_seed_features'][seed]
        assert list(features['current'][c][seed]) == list(ref), \
            f"{c} seed {seed}: 현행 방식 feature 가 screening_results.pkl 과 다름 — 중단"
        a, b   = features['current'][c][seed], features['alternative'][c][seed]
        shared = set(a) & set(b)
        fs_rows.append({'condition': c, 'condition_label': COND_LABELS[c], 'seed': seed,
                        'n_current': len(a), 'n_alternative': len(b), 'n_shared': len(shared),
                        'jaccard': len(shared) / len(set(a) | set(b)),
                        'only_current':     ', '.join(sorted(short(g) for g in set(a) - set(b))),
                        'only_alternative': ', '.join(sorted(short(g) for g in set(b) - set(a)))})
df_fs = pd.DataFrame(fs_rows)

for s in SCHEMES:
    for seed in SEEDS:
        assert features[s]['cond1'][seed] == fc[s][seed][0]
print("current scheme features = screening_results.pkl (4 conditions × 10 seeds)")
print()
print("=== Feature selection: current vs alternative zero replacement ===")
print(df_fs.groupby(['condition', 'condition_label'])
      .agg(mean_jaccard=('jaccard', 'mean'), mean_shared=('n_shared', 'mean'),
           n_current=('n_current', lambda x: f"{x.min()}-{x.max()}"),
           n_alternative=('n_alternative', lambda x: f"{x.min()}-{x.max()}"))
      .round(3).to_string())

current scheme features = screening_results.pkl (4 conditions × 10 seeds)

=== Feature selection: current vs alternative zero replacement ===
                              mean_jaccard  mean_shared n_current n_alternative
condition condition_label                                                      
cond1     Optuna × FC Top10          0.577          7.3     10-10         10-10
cond2     Optuna × VIP≥1.0           0.590         17.4     22-28         21-24
cond3     K-means × FC Top10         0.577          7.3     10-10         10-10
cond4     K-means × VIP≥1.0          0.562         19.6     27-31         23-30


## 7. Final model (Condition 1 × Random Forest) — CV test and holdout

Same procedure as the main pipeline (GridSearchCV, cv = 5): classification as in `4. classification.ipynb`,
regression on Early + Intermediate samples as in `5. regression.ipynb`, and end-to-end holdout evaluation (n = 27) as in `6. validation.ipynb`.

Checks: the current scheme must reproduce the stored test results (`sensitivity_results.pkl` / `reg_sensitivity_results.pkl`, label seed 42) for all 10 seeds,
and the stored seed-42 holdout result (`validation_results.pkl`).

In [10]:
def fit_eval(sp, feats):
    tr, te, va = sp['train'], sp['test'], sp['val']

    clf = GridSearchCV(RandomForestClassifier(random_state=42), RF_GRID, cv=5,
                       scoring='f1_macro', n_jobs=-1, refit=True).fit(
        tr[feats].values, tr['Label'].values).best_estimator_
    tr_fm, te_fm = tr[tr['Label'] != 1], te[te['Label'] != 1]
    reg = GridSearchCV(RandomForestRegressor(random_state=42), RF_GRID, cv=5,
                       scoring='neg_mean_absolute_error', n_jobs=-1, refit=True).fit(
        tr_fm[feats].values, tr_fm[tpc_col].values).best_estimator_

    y_te = clf.predict(te[feats].values)
    r_te = reg.predict(te_fm[feats].values)

    y_va   = clf.predict(va[feats].values)
    va_reg = va[pd.Series(y_va).isin([0, 2]).values]
    r_va   = reg.predict(va_reg[feats].values)

    return {'clf': clf, 'reg': reg, 'features': feats,
            'test_acc': accuracy_score(te['Label'].values, y_te),
            'test_f1':  f1_score(te['Label'].values, y_te, average='macro'),
            'test_r2':  r2_score(te_fm[tpc_col].values, r_te),
            'test_mae': mean_absolute_error(te_fm[tpc_col].values, r_te),
            'ho_acc':   accuracy_score(va['Label'].values, y_va),
            'ho_f1':    f1_score(va['Label'].values, y_va, average='macro', zero_division=0),
            'ho_r2':    r2_score(va_reg[tpc_col].values, r_va),
            'ho_mae':   mean_absolute_error(va_reg[tpc_col].values, r_va),
            'ho_reg_n': len(va_reg)}


models = {s: {} for s in SCHEMES}
for s in SCHEMES:
    for seed in SEEDS:
        models[s][seed] = fit_eval(splits[s][seed], fc[s][seed][0])
        m = models[s][seed]
        print(f"{s:12s} seed={seed} | test acc={m['test_acc']:.3f} f1={m['test_f1']:.3f} "
              f"R2={m['test_r2']:.3f} MAE={m['test_mae']:.3f} | holdout acc={m['ho_acc']:.3f} "
              f"f1={m['ho_f1']:.3f} R2={m['ho_r2']:.3f} MAE={m['ho_mae']:.3f}")

# ── 현행 방식 재현 확인 ───────────────────────────────────────────
chk = []
for seed in SEEDS:
    m = models['current'][seed]
    chk += [('test_acc', seed, m['test_acc'] - sens_clf[LABEL_SEED][seed]['acc']),
            ('test_f1',  seed, m['test_f1']  - sens_clf[LABEL_SEED][seed]['f1']),
            ('test_r2',  seed, m['test_r2']  - sens_reg[LABEL_SEED][seed]['r2']),
            ('test_mae', seed, m['test_mae'] - sens_reg[LABEL_SEED][seed]['mae'])]
m42 = models['current'][LABEL_SEED]
chk += [('holdout_acc', LABEL_SEED, m42['ho_acc'] - val_res['clf_acc']),
        ('holdout_f1',  LABEL_SEED, m42['ho_f1']  - val_res['clf_f1']),
        ('holdout_r2',  LABEL_SEED, m42['ho_r2']  - val_res['e2e_r2']),
        ('holdout_mae', LABEL_SEED, m42['ho_mae'] - val_res['e2e_mae'])]
df_checks_model = pd.DataFrame(chk, columns=['metric', 'seed', 'difference'])
max_diff_model = float(df_checks_model['difference'].abs().max())
print(f"\ncurrent scheme vs stored main-pipeline results: max |difference| = {max_diff_model:.2e}")
assert max_diff_model < 1e-9, "현행 방식이 저장된 결과를 재현하지 못함 — 비교 불가"

# ── 비교: paired t-test ───────────────────────────────────────────
perf_rows, perf_seed = [], []
for ev, keys in (('CV test', (('Accuracy', 'test_acc'), ('Macro-F1', 'test_f1'),
                              ('R2', 'test_r2'), ('MAE (log CFU/g)', 'test_mae'))),
                 ('Holdout', (('Accuracy', 'ho_acc'), ('Macro-F1', 'ho_f1'),
                              ('R2', 'ho_r2'), ('MAE (log CFU/g)', 'ho_mae')))):
    for metric, k in keys:
        a = np.array([models['current'][s][k]     for s in SEEDS])
        b = np.array([models['alternative'][s][k] for s in SEEDS])
        t, p = stats.ttest_rel(a, b)
        perf_rows.append({'evaluation': ev, 'metric': metric,
                          'current_mean': a.mean(), 'current_sd': a.std(),
                          'alternative_mean': b.mean(), 'alternative_sd': b.std(),
                          'delta': b.mean() - a.mean(), 't': t, 'p_paired': p,
                          'significant_a0.05': 'yes' if p < 0.05 else 'no'})
        for s_, va_, vb_ in zip(SEEDS, a, b):
            perf_seed.append({'evaluation': ev, 'metric': metric, 'seed': s_,
                              'current': va_, 'alternative': vb_})
df_perf      = pd.DataFrame(perf_rows)
df_perf_seed = pd.DataFrame(perf_seed)

print()
print("=== Condition 1 × RF: current vs alternative zero replacement (10 seeds, paired t-test) ===")
print(df_perf.round(4).to_string(index=False))

current      seed=38 | test acc=0.870 f1=0.865 R2=0.748 MAE=0.386 | holdout acc=0.963 f1=0.972 R2=0.690 MAE=0.417
current      seed=39 | test acc=0.833 f1=0.795 R2=0.695 MAE=0.446 | holdout acc=0.926 f1=0.944 R2=0.670 MAE=0.528
current      seed=40 | test acc=0.889 f1=0.899 R2=0.645 MAE=0.471 | holdout acc=0.889 f1=0.864 R2=0.565 MAE=0.483
current      seed=41 | test acc=0.852 f1=0.840 R2=0.735 MAE=0.380 | holdout acc=0.815 f1=0.767 R2=0.847 MAE=0.370
current      seed=42 | test acc=0.852 f1=0.867 R2=0.685 MAE=0.462 | holdout acc=0.815 f1=0.767 R2=0.757 MAE=0.471
current      seed=43 | test acc=0.815 f1=0.767 R2=0.697 MAE=0.446 | holdout acc=0.815 f1=0.752 R2=0.627 MAE=0.443
current      seed=44 | test acc=0.889 f1=0.885 R2=0.784 MAE=0.369 | holdout acc=0.889 f1=0.854 R2=0.529 MAE=0.562
current      seed=45 | test acc=0.815 f1=0.811 R2=0.705 MAE=0.427 | holdout acc=0.889 f1=0.802 R2=0.728 MAE=0.419
current      seed=46 | test acc=0.870 f1=0.853 R2=0.728 MAE=0.481 | holdout acc=0.667 f1

## 8. All conditions × all models — CV test performance

Same training as `4. classification.ipynb` and `5. regression.ipynb` Section 3 (GridSearchCV, cv = 5, same grids and random states),
on the alternative-scheme CLR values and features of Section 6. Current-scheme results are read from `classification_results.pkl` and `regression_results.pkl`.
Results are saved after each block so that the comparison can be re-run without retraining.

The best model per condition is selected as in `4. classification.ipynb` Section 5 (highest mean test macro-F1) and, for regression, by the highest mean test R².
The Condition 1 × RF results must equal those of Section 7.

In [11]:
GRID_PKL = f"{PKL_DIR}/zero_replacement_grid.pkl"
grid_results = {'features': features}


def _save_grid():
    with open(GRID_PKL, 'wb') as f:
        pickle.dump(grid_results, f)
    print(f"Saved → {GRID_PKL}")


# ══════════════════ Classification (alternative) ══════════════════
alt_clf = {c: {m: {} for m in clf_configs} for c in CONDS}
for c in CONDS:
    sp_all = splits_by_labeling['alternative'][COND_LABELING[c]]
    print(f"\n{'='*60}\nCondition: {COND_LABELS[c]}\n{'='*60}")
    for seed in SEEDS:
        feats = features['alternative'][c][seed]
        tr, te = sp_all[seed]['train'], sp_all[seed]['test']
        for model_name, cfg in clf_configs.items():
            grid = GridSearchCV(_base.clone(cfg['estimator']), cfg['params'], cv=5,
                                scoring='f1_macro', n_jobs=-1, refit=True)
            grid.fit(tr[feats].values, tr['Label'].values)
            y_pred = grid.best_estimator_.predict(te[feats].values)
            alt_clf[c][model_name][seed] = {
                'test_acc': accuracy_score(te['Label'].values, y_pred),
                'test_f1':  f1_score(te['Label'].values, y_pred, average='macro'),
                'best_params': grid.best_params_, 'features': feats}
        print(f"  seed={seed} | " + " | ".join(
            f"{m}(acc={alt_clf[c][m][seed]['test_acc']:.2f} f1={alt_clf[c][m][seed]['test_f1']:.2f})"
            for m in clf_configs))

grid_results['classification_alternative'] = alt_clf
_save_grid()


Condition: Optuna × FC Top10
  seed=38 | RandomForest(acc=0.85 f1=0.85) | XGBoost(acc=0.81 f1=0.81) | SVC(acc=0.87 f1=0.87) | LogisticRegression(acc=0.83 f1=0.83) | KNN(acc=0.85 f1=0.85)
  seed=39 | RandomForest(acc=0.78 f1=0.71) | XGBoost(acc=0.74 f1=0.69) | SVC(acc=0.81 f1=0.78) | LogisticRegression(acc=0.80 f1=0.75) | KNN(acc=0.85 f1=0.81)
  seed=40 | RandomForest(acc=0.85 f1=0.86) | XGBoost(acc=0.83 f1=0.82) | SVC(acc=0.89 f1=0.90) | LogisticRegression(acc=0.83 f1=0.81) | KNN(acc=0.83 f1=0.86)
  seed=41 | RandomForest(acc=0.83 f1=0.79) | XGBoost(acc=0.87 f1=0.86) | SVC(acc=0.89 f1=0.85) | LogisticRegression(acc=0.89 f1=0.88) | KNN(acc=0.81 f1=0.78)
  seed=42 | RandomForest(acc=0.74 f1=0.71) | XGBoost(acc=0.76 f1=0.71) | SVC(acc=0.74 f1=0.71) | LogisticRegression(acc=0.72 f1=0.69) | KNN(acc=0.83 f1=0.81)
  seed=43 | RandomForest(acc=0.76 f1=0.71) | XGBoost(acc=0.78 f1=0.72) | SVC(acc=0.81 f1=0.77) | LogisticRegression(acc=0.83 f1=0.79) | KNN(acc=0.74 f1=0.71)
  seed=44 | RandomFore

In [12]:
# ══════════════════ Regression (alternative, Early + Intermediate) ══════════════════
alt_reg = {c: {m: {} for m in reg_configs} for c in CONDS}
for c in CONDS:
    sp_all = splits_by_labeling['alternative'][COND_LABELING[c]]
    print(f"\n{'='*60}\nCondition: {COND_LABELS[c]}\n{'='*60}")
    for seed in SEEDS:
        feats = features['alternative'][c][seed]
        tr, te = sp_all[seed]['train'], sp_all[seed]['test']
        tr_fm, te_fm = tr[tr['Label'] != 1], te[te['Label'] != 1]
        for model_name, cfg in reg_configs.items():
            grid = GridSearchCV(_base.clone(cfg['estimator']), cfg['params'], cv=5,
                                scoring='neg_mean_absolute_error', n_jobs=-1, refit=True)
            grid.fit(tr_fm[feats].values, tr_fm[tpc_col].values)
            y_pred = grid.best_estimator_.predict(te_fm[feats].values)
            alt_reg[c][model_name][seed] = {
                'test_r2':  r2_score(te_fm[tpc_col].values, y_pred),
                'test_mae': mean_absolute_error(te_fm[tpc_col].values, y_pred),
                'best_params': grid.best_params_, 'features': feats}
        print(f"  seed={seed} | " + " | ".join(
            f"{m} R²={alt_reg[c][m][seed]['test_r2']:.3f}" for m in reg_configs))

grid_results['regression_alternative'] = alt_reg
_save_grid()


Condition: Optuna × FC Top10


I0000 00:00:1794853239.547948 2831127 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794853239.547948 2831119 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794853239.557777 2831126 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794853239.559722 2831111 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point ro

  seed=38 | LightGBM R²=0.512 | RandomForest R²=0.558 | XGBoost R²=0.514 | SVR R²=0.539 | LSTM R²=0.520


I0000 00:00:1794853289.469064 3389304 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794853289.522124 3520247 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794853289.581294 3476506 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794853289.581295 3522119 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point ro

  seed=39 | LightGBM R²=0.636 | RandomForest R²=0.687 | XGBoost R²=0.710 | SVR R²=0.679 | LSTM R²=0.715


I0000 00:00:1794853332.957533   31103 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794853332.974303 4185211 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794853332.974931   31104 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794853333.031210 4161471 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point ro

... (71 outputs left out)

I0000 00:00:1794855058.434487 4137476 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794855058.451638 4178733 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794855058.451638 4185605 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794855058.454976 4178724 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point ro

  seed=46 | LightGBM R²=0.331 | RandomForest R²=0.314 | XGBoost R²=0.303 | SVR R²=0.423 | LSTM R²=0.332


I0000 00:00:1794855106.124206  681466 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794855106.124206  681432 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794855106.124206  681470 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794855106.151881  621066 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point ro

  seed=47 | LightGBM R²=0.413 | RandomForest R²=0.464 | XGBoost R²=0.441 | SVR R²=0.317 | LSTM R²=0.318
Saved → output/pkl/zero_replacement_grid.pkl


In [13]:
# ── 일치 확인: grid 의 Cond 1 × RF 는 Section 7 과 같아야 한다 ────────
for seed in SEEDS:
    for k, res in (('test_acc', alt_clf), ('test_f1', alt_clf), ('test_r2', alt_reg), ('test_mae', alt_reg)):
        d = abs(res[FINAL_COND][FINAL_MODEL][seed][k] - models['alternative'][seed][k])
        assert d < 1e-12, f"seed {seed} {k}: grid 와 Section 7 결과가 다름"
    for k, res in (('test_acc', stored_clf), ('test_f1', stored_clf), ('test_r2', stored_reg), ('test_mae', stored_reg)):
        d = abs(res[FINAL_COND][FINAL_MODEL][seed][k] - models['current'][seed][k])
        assert d < 1e-12, f"seed {seed} {k}: 저장된 결과와 Section 7 현행 결과가 다름"
print("Cond 1 × RF: grid = Section 7 (both schemes)")

# ── 비교: 20 조합 × 2 task, paired t-test ────────────────────────────
grid_rows, grid_raw = [], []
for task, cur, alt, cfgs, metrics in (
        ('Classification', stored_clf, alt_clf, clf_configs, (('Accuracy', 'test_acc'), ('Macro-F1', 'test_f1'))),
        ('Regression',     stored_reg, alt_reg, reg_configs, (('R2', 'test_r2'), ('MAE (log CFU/g)', 'test_mae')))):
    for c in CONDS:
        for m in cfgs:
            for metric, k in metrics:
                a = np.array([cur[c][m][s][k] for s in SEEDS])
                b = np.array([alt[c][m][s][k] for s in SEEDS])
                t, p = stats.ttest_rel(a, b)
                grid_rows.append({'task': task, 'condition': c, 'condition_label': COND_LABELS[c],
                                  'model': m, 'metric': metric,
                                  'current_mean': a.mean(), 'current_sd': a.std(),
                                  'alternative_mean': b.mean(), 'alternative_sd': b.std(),
                                  'delta': b.mean() - a.mean(), 't': t, 'p_paired': p,
                                  'significant_a0.05': 'yes' if p < 0.05 else 'no'})
                for s_, va_, vb_ in zip(SEEDS, a, b):
                    grid_raw.append({'task': task, 'condition': c, 'model': m, 'metric': metric,
                                     'seed': s_, 'current': va_, 'alternative': vb_})
df_grid      = pd.DataFrame(grid_rows)
df_grid_seed = pd.DataFrame(grid_raw)

best_rows = []
for task, cur, alt, cfgs, k in (('Classification', stored_clf, alt_clf, clf_configs, 'test_f1'),
                                ('Regression',     stored_reg, alt_reg, reg_configs, 'test_r2')):
    for c in CONDS:
        row = {'task': task, 'condition': c, 'condition_label': COND_LABELS[c]}
        for scheme, res in (('current', cur), ('alternative', alt)):
            means = {m: np.mean([res[c][m][s][k] for s in SEEDS]) for m in cfgs}
            best  = max(means, key=means.get)
            row[f'{scheme}_best_model'] = best
            row[f'{scheme}_best_mean_{k}'] = means[best]
        row['same_best_model'] = row['current_best_model'] == row['alternative_best_model']
        best_rows.append(row)
df_grid_best = pd.DataFrame(best_rows)

pd.set_option('display.width', 220)
for task in ('Classification', 'Regression'):
    print(f"\n=== {task}: current vs alternative zero replacement (CV test, 10 seeds) ===")
    print(df_grid[df_grid['task'] == task].drop(columns=['task', 'condition_label']).round(4).to_string(index=False))
print("\n=== Best model per condition ===")
print(df_grid_best.round(4).to_string(index=False))

grid_results.update({'performance': df_grid, 'performance_per_seed': df_grid_seed,
                     'best_model_by_condition': df_grid_best})
_save_grid()

Cond 1 × RF: grid = Section 7 (both schemes)

=== Classification: current vs alternative zero replacement (CV test, 10 seeds) ===
condition              model   metric  current_mean  current_sd  alternative_mean  alternative_sd   delta       t  p_paired significant_a0.05
    cond1       RandomForest Accuracy        0.8574      0.0275            0.8019          0.0423 -0.0556  5.8095    0.0003               yes
    cond1       RandomForest Macro-F1        0.8476      0.0419            0.7788          0.0567 -0.0688  5.5945    0.0003               yes
    cond1            XGBoost Accuracy        0.8074      0.0323            0.7944          0.0356 -0.0130  1.2104    0.2569                no
    cond1            XGBoost Macro-F1        0.7884      0.0486            0.7747          0.0515 -0.0137  1.5321    0.1599                no
    cond1                SVC Accuracy        0.8463      0.0406            0.8167          0.0680 -0.0296  2.3881    0.0407               yes
    cond1         

## 9. SHAP rankings of the final model

Same procedures as the main pipeline, for each seed and both zero replacements:

| Analysis | Samples | Source procedure |
|---|---|---|
| Classification (test) | test set | `4. classification.ipynb` (sum of per-class mean \|SHAP\|) |
| Regression (test) | test set, Early + Intermediate | `5. regression.ipynb` |
| Classification (validation) | holdout set | `6. validation.ipynb` (mean \|SHAP\| over samples and classes) |
| Regression (validation) | holdout set, Early + Intermediate | `6. validation.ipynb` |

Ranks are taken within each seed's own Top 10 features. Because the two zero replacements select partly different genera (Section 5),
the SHAP rankings differ both because the feature set changes and because importance within shared genera changes (Sections 5–6);
`n_seeds_selected` shows how often each genus is in the feature set.
The top-10 tables list the genera selected in at least 5 of the 10 seeds, ordered by their mean rank over the seeds in which they were selected; genera selected in fewer seeds are listed in the `SHAP_excluded` sheet.

In [14]:
def _shap_clf_sum(clf, feats, X):
    sv = shap.TreeExplainer(clf).shap_values(pd.DataFrame(X[feats].values, columns=[short(g) for g in feats]))
    if isinstance(sv, np.ndarray) and sv.ndim == 3:
        sv = [sv[:, :, i] for i in range(sv.shape[2])]
    cls = list(clf.classes_)
    return sum(np.mean(np.abs(sv[cls.index(l)]), axis=0) for l in LABEL_ORDER)


def _shap_clf_mean(clf, feats, X):
    sv = np.array(shap.TreeExplainer(clf).shap_values(pd.DataFrame(X[feats].values, columns=[short(g) for g in feats])))
    n_cls = len(clf.classes_)
    if sv.ndim == 3 and sv.shape[0] == n_cls:
        return np.mean(np.abs(sv), axis=(0, 1))
    if sv.ndim == 3 and sv.shape[2] == n_cls:
        return np.mean(np.abs(sv), axis=(0, 2))
    return np.mean(np.abs(sv), axis=0)


def _shap_reg(reg, feats, X):
    X_fm = X[X['Label'] != 1]
    sv = np.array(shap.TreeExplainer(reg).shap_values(pd.DataFrame(X_fm[feats].values, columns=[short(g) for g in feats])))
    if sv.ndim == 1:
        sv = sv.reshape(1, -1)
    return np.mean(np.abs(sv), axis=0)


ANALYSES = {
    'Classification (test)':       lambda m, sp: _shap_clf_sum(m['clf'],  m['features'], sp['test']),
    'Regression (test)':           lambda m, sp: _shap_reg(m['reg'],      m['features'], sp['test']),
    'Classification (validation)': lambda m, sp: _shap_clf_mean(m['clf'], m['features'], sp['val']),
    'Regression (validation)':     lambda m, sp: _shap_reg(m['reg'],      m['features'], sp['val']),
}

shap_long = []
for s in SCHEMES:
    for seed in SEEDS:
        m, sp = models[s][seed], splits[s][seed]
        names = [short(g) for g in m['features']]
        for analysis, fn in ANALYSES.items():
            imp  = pd.Series(fn(m, sp), index=names)
            rank = imp.rank(ascending=False, method='min').astype(int)
            for g in names:
                shap_long.append({'scheme': s, 'analysis': analysis, 'seed': seed, 'genus': g,
                                  'mean_abs_shap': imp[g], 'rank': rank[g]})
df_shap_long = pd.DataFrame(shap_long)

df_shap_summary = (df_shap_long.groupby(['analysis', 'scheme', 'genus'])['rank']
                   .agg(n_seeds_selected='count', mean_rank='mean',
                        sd_rank=lambda x: np.std(x),
                        n_rank1=lambda x: int((x == 1).sum()),
                        n_top3=lambda x: int((x <= 3).sum()))
                   .reset_index()
                   .sort_values(['analysis', 'scheme', 'mean_rank']))

# ── 분석별 상위 10 나란히: 10 seed 중 MIN_SEEDS 개 이상에서 선택된 genus 만, 선택된 seed 들의 평균 순위로 정렬 ──
MIN_SEEDS = 5
side_rows, excl_rows = [], []
for analysis in ANALYSES:
    tops = {}
    for s in SCHEMES:
        d = df_shap_summary[(df_shap_summary['analysis'] == analysis) & (df_shap_summary['scheme'] == s)]
        tops[s] = d[d['n_seeds_selected'] >= MIN_SEEDS].head(10).reset_index(drop=True)
        for r in d[d['n_seeds_selected'] < MIN_SEEDS].itertuples():
            excl_rows.append({'analysis': analysis, 'scheme': s, 'genus': r.genus,
                              'n_seeds_selected': r.n_seeds_selected, 'mean_rank': r.mean_rank})
    for i in range(10):
        row = {'analysis': analysis, 'position': i + 1}
        for s in SCHEMES:
            r = tops[s].iloc[i] if i < len(tops[s]) else None
            row[f'{s}_genus']     = r['genus'] if r is not None else ''
            row[f'{s}_mean_rank'] = r['mean_rank'] if r is not None else np.nan
            row[f'{s}_n_seeds']   = r['n_seeds_selected'] if r is not None else np.nan
        side_rows.append(row)
df_shap_top10 = pd.DataFrame(side_rows)
df_shap_excluded = pd.DataFrame(excl_rows)

# ── 순위 안정성: seed 별 1위 ──────────────────────────────────────
stab_rows = []
for analysis in ANALYSES:
    for s in SCHEMES:
        d = df_shap_long[(df_shap_long['analysis'] == analysis) & (df_shap_long['scheme'] == s)]
        top1 = d[d['rank'] == 1].groupby('seed')['genus'].first()
        vc = top1.value_counts()
        stab_rows.append({'analysis': analysis, 'scheme': s,
                          'n_distinct_top1': int(vc.size),
                          'most_frequent_top1': vc.index[0], 'n_seeds_most_frequent_top1': int(vc.iloc[0]),
                          'top1_by_seed': ', '.join(f"{k}:{v}" for k, v in top1.items())})
df_shap_stability = pd.DataFrame(stab_rows)

# ── 본문 핵심 taxa + 리뷰어가 언급한 genus ────────────────────────
watch = KEY_TAXA + [g for g in REVIEWER_RANKS if g not in KEY_TAXA]
df_shap_watch = (df_shap_summary[df_shap_summary['genus'].isin(watch)]
                 .pivot_table(index=['analysis', 'genus'], columns='scheme',
                              values=['n_seeds_selected', 'mean_rank'])
                 .round(2))

pd.set_option('display.width', 220)
for analysis in ANALYSES:
    print(f"\n=== SHAP top 10 (genera selected in ≥ {MIN_SEEDS} of 10 seeds; mean rank over the seeds in which selected) — {analysis} ===")
    print(df_shap_top10[df_shap_top10['analysis'] == analysis]
          .drop(columns='analysis').round(2).to_string(index=False))
print(f"\n=== Genera selected in < {MIN_SEEDS} seeds (not listed above) ===")
print(df_shap_excluded.round(2).to_string(index=False))
print("\n=== Rank stability: top-1 genus per seed ===")
print(df_shap_stability.to_string(index=False))
print("\n=== Key taxa and genera named by the reviewer ===")
print(df_shap_watch.to_string())


=== SHAP top 10 (genera selected in ≥ 5 of 10 seeds; mean rank over the seeds in which selected) — Classification (test) ===
 position          current_genus  current_mean_rank  current_n_seeds      alternative_genus  alternative_mean_rank  alternative_n_seeds
        1             Iodobacter                1.0               10            Brochothrix                   1.20                   10
        2             Vagococcus                2.3               10             Iodobacter                   1.90                   10
        3          Streptococcus                3.8               10      Janthinobacterium                   3.50                   10
        4 Hafnia-Obesumbacterium                4.3               10             Vagococcus                   4.00                   10
        5         Carnobacterium                5.2               10 Hafnia-Obesumbacterium                   4.80                   10
        6               Bacillus                6.6       

## 10. Save

In [15]:
zero_replacement_results = {
    'schemes': SCHEMES,
    'description': df_desc, 'nondetection_by_genus': df_nondetect,
    'fc_nondetection_main': df_fc_nondetect,
    'fc_rank_seed42': df_fc_rank42, 'fc_top20_seed42': df_fc_top20,
    'fc_reviewer_ranks': df_fc_reviewer, 'fc_top10_all_seeds': df_fc_top10,
    'fc_top10_nondetection': df_fc_diag,
    'features': features, 'feature_selection': df_fs,
    'grid_performance': df_grid, 'grid_performance_per_seed': df_grid_seed,
    'grid_best_model_by_condition': df_grid_best,
    'performance': df_perf, 'performance_per_seed': df_perf_seed,
    'shap_long': df_shap_long, 'shap_summary': df_shap_summary,
    'shap_top10': df_shap_top10, 'shap_stability': df_shap_stability,
    'checks': {'clr_max_abs_diff': max_diff_clr, 'model_checks': df_checks_model},
}
with open(f"{PKL_DIR}/zero_replacement_results.pkl", 'wb') as f:
    pickle.dump(zero_replacement_results, f)
print(f"Saved → {PKL_DIR}/zero_replacement_results.pkl")

xlsx = f"{OUTPUT_DIR}/zero_replacement_test.xlsx"
with pd.ExcelWriter(xlsx, engine='openpyxl') as w:
    df_desc.to_excel(w,           sheet_name='Data_description',       index=False)
    df_nondetect.to_excel(w,      sheet_name='Nondetection_by_genus',  index=False)
    df_fc_nondetect.to_excel(w,   sheet_name='FC_top10_main_nondetect', index=False)
    df_fc_top20.to_excel(w,       sheet_name='FC_seed42_top20',        index=False)
    df_fc_rank42.to_excel(w,      sheet_name='FC_seed42_all_genera',   index=False)
    df_fc_reviewer.to_excel(w,    sheet_name='FC_reviewer_ranks',      index=False)
    df_fc_top10.to_excel(w,       sheet_name='FC_top10_all_seeds',     index=False)
    df_fc_diag.to_excel(w,        sheet_name='FC_top10_nondetection',  index=False)
    df_fs.to_excel(w,             sheet_name='Feature_selection_all',  index=False)
    df_perf.to_excel(w,           sheet_name='Final_model_performance', index=False)
    df_perf_seed.to_excel(w,      sheet_name='Final_model_per_seed',   index=False)
    df_grid[df_grid['task'] == 'Classification'].to_excel(w, sheet_name='Perf_classification', index=False)
    df_grid[df_grid['task'] == 'Regression'].to_excel(w,     sheet_name='Perf_regression',     index=False)
    df_grid_best.to_excel(w,      sheet_name='Best_model_by_condition', index=False)
    df_grid_seed.to_excel(w,      sheet_name='Perf_per_seed_raw',      index=False)
    df_shap_top10.to_excel(w,     sheet_name='SHAP_top10',             index=False)
    df_shap_excluded.to_excel(w,  sheet_name='SHAP_excluded',          index=False)
    df_shap_stability.to_excel(w, sheet_name='SHAP_rank_stability',    index=False)
    df_shap_summary.to_excel(w,   sheet_name='SHAP_summary',           index=False)
    df_shap_long.to_excel(w,      sheet_name='SHAP_rank_per_seed',     index=False)
    df_checks_model.to_excel(w,   sheet_name='Checks',                 index=False)
print(f"Excel saved → {xlsx}")

Saved → output/pkl/zero_replacement_results.pkl
Excel saved → output/zero_replacement_test.xlsx
